# 🧠 ResNet Cookbook: Understanding Residual Learning & Skip Connections

**Experiment 5 — Deep Learning Lab**

---

## 📋 Learning Objectives

By the end of this cookbook, you will be able to:

1. **Explain** the vanishing gradient problem and why it occurs in deep networks
2. **Derive** the residual learning formulation $y = \mathcal{F}(x) + x$ and its gradient properties
3. **Compare** four network configurations and interpret their training dynamics
4. **Prove** — with per-layer evidence — that skip connections fix gradient flow

## 🔬 Experiment Overview

We train **four** model variants on the **SVHN** (Street View House Numbers) dataset:

| # | Configuration | Residual Blocks? | Skip (`F(x)+x`)? |
|---|--------------|:-:|:-:|
| 1 | Simple CNN | ❌ | ❌ |
| 2 | Simple CNN + Skip | ❌ | ✅ |
| 3 | ResNet (no skip) | ✅ | ❌ |
| 4 | ResNet (with skip) | ✅ | ✅ |

All models share the same depth (~20 layers), dataset, hyperparameters, and initialization.

---
# Part I: Theory
---

## 1. The Vanishing Gradient Problem

### Why Deep Networks Struggle

Neural networks learn by **backpropagation**. For a network with $L$ layers, the gradient at layer $l$ is:

$$\frac{\partial \mathcal{L}}{\partial W_l} = \frac{\partial \mathcal{L}}{\partial a_L} \cdot \prod_{i=l}^{L-1} \frac{\partial a_{i+1}}{\partial a_i}$$

### The Core Issue

Each Jacobian factor $\frac{\partial a_{i+1}}{\partial a_i}$ depends on the activation function and weights:

- **Sigmoid/Tanh**: Derivatives bounded in $(0, 0.25]$ / $(0, 1]$
- **ReLU**: 0 or 1, but weight matrices can still cause shrinkage

When many factors < 1, the product **exponentially decays**:

$$\prod_{i=l}^{L-1} \frac{\partial a_{i+1}}{\partial a_i} \approx \alpha^{L-l}, \quad |\alpha| < 1$$

**Result**: Early layers receive tiny gradients → tiny weight updates → they stop learning.

### Visual Intuition

Think of it like a game of telephone — the error signal starts strong at the output but gets **weakened** as it passes backward through each layer. By the time it reaches the first layers, it's barely a whisper.

> **Key Insight**: The deeper the network, the worse this problem gets.

## 2. The Residual Learning Framework (He et al., 2015)

### The Big Idea

Instead of learning the desired mapping $\mathcal{H}(x)$ directly, learn the **residual**:

$$\mathcal{F}(x) = \mathcal{H}(x) - x \quad \Rightarrow \quad y = \mathcal{F}(x) + x$$

- $\mathcal{F}(x)$ = what the layers learn (the **residual**)
- $x$ = the **identity shortcut** (skip connection)

### Why Learning Residuals is Easier

If the optimal mapping is close to identity:
- **Without skip**: Must learn $\mathcal{H}(x) \approx x$ — complex through nonlinear layers
- **With skip**: Only learn $\mathcal{F}(x) \approx 0$ — pushing weights toward zero is trivial!

### Residual Block Structure

```
Input (x)
  │
  ├──────────────────┐
  │                  │ (skip / identity)
  ▼                  │
Conv → BN → ReLU    │
  │                  │
  ▼                  │
Conv → BN            │
  │                  │
  ▼                  │
  + ◄────────────────┘  ← Element-wise addition
  │
  ▼
ReLU → Output (y = F(x) + x)
```

When dimensions differ, a **1×1 conv projection shortcut** is used: $y = \mathcal{F}(x) + W_s \cdot x$

## 3. Why Skip Connections Fix Gradient Flow

### The Mathematical Proof

For $y = \mathcal{F}(x) + x$, the backward gradient is:

$$\frac{\partial y}{\partial x} = \frac{\partial \mathcal{F}(x)}{\partial x} + \mathbf{I}$$

### Why This Changes Everything

**Plain network** chain rule:
$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{i=l}^{L-1} \frac{\partial \mathcal{F}_i}{\partial x_i} \quad \text{→ can vanish}$$

**Residual network** chain rule:
$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{i=l}^{L-1} \left(\frac{\partial \mathcal{F}_i}{\partial x_i} + \mathbf{I}\right) \quad \text{→ stays healthy}$$

The $+\mathbf{I}$ ensures **even if $\frac{\partial \mathcal{F}_i}{\partial x_i}$ is small, the gradient still flows.**

### Comparison Table

| Aspect | Plain Network | Residual Network |
|--------|:-:|:-:|
| Gradient path | Single road through layers | Highway + local roads |
| If one layer blocks | All upstream layers starve | Highway keeps flowing |
| Early layer updates | Exponentially weaker | Maintained by identity path |
| Gradient magnitude | $O(\alpha^L)$, can vanish | $O(1)$, stays healthy |

> **Key Takeaway**: Skip connections provide a **mathematical guarantee** that gradients can flow to any layer.

---
# Part II: Experiments
---

## 4. Experiment Setup

### Dataset: SVHN (Street View House Numbers)
- **Task**: Classify 32×32 RGB images of digits (0–9)
- **Training**: 73,257 images | **Test**: 26,032 images

### Common Hyperparameters

| Parameter | Value |
|-----------|-------|
| Batch Size | 128 |
| Epochs | 20 |
| Optimizer | Adam (lr=1e-3) |
| Scheduler | StepLR (step=10, γ=0.5) |
| Loss | CrossEntropyLoss |
| Init | Kaiming Normal |

### Metrics Tracked Per Conv Layer

| Metric | Tells Us |
|--------|----------|
| **Gradient Norm (L2)** | How strong is the learning signal? |
| **Weight Norm (L2)** | Is capacity being utilized? |
| **Weight Delta (L2)** | How much did weights change this epoch? |
| **Gradient Ratio** | First/last layer grad — is flow balanced? |

> Low gradient norm + low weight delta = layer is **not learning** = vanishing gradients.

---
## 5. Case 1: Simple CNN — No Residual, No Skip

### Architecture
Plain stack of `Conv → BN → ReLU`. No residual structure, no shortcuts.

```
Input (3×32×32) → Conv stem (64ch)
→ 3× Conv-BN-ReLU (64ch, 32×32)
→ 3× Conv-BN-ReLU (128ch, 16×16)
→ 3× Conv-BN-ReLU (256ch, 8×8)
→ AvgPool → FC(256→10)
```

### Expected: Early layers show weaker gradients and updates than later layers.

In [ ]:
# Simple CNN Model (No Residual, No Skip)
import torch.nn as nn

class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = ConvBnRelu(3, 64)
        channels = [64,64,64, 128,128,128, 256,256,256]
        layers, in_ch = [], 64
        for idx, out_ch in enumerate(channels):
            stride = 2 if idx in (3, 6) else 1
            layers.append(ConvBnRelu(in_ch, out_ch, stride=stride))
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.stem(x); x = self.features(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))

### Results: Simple CNN


**📈 Accuracy Curves**
![](../wo_resnet_wo_skip/plots/accuracy_curves.png)


**📉 Loss Curves**
![](../wo_resnet_wo_skip/plots/loss_curves.png)


**🔬 Epoch Gradient Norms**
![](../wo_resnet_wo_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../wo_resnet_wo_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../wo_resnet_wo_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../wo_resnet_wo_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../wo_resnet_wo_skip/plots/weight_deltas.png)


**🎯 Confusion Matrix**
![](../wo_resnet_wo_skip/plots/confusion_matrix.png)


### 🔍 Observations — Simple CNN

1. **Gradient Vanishing**: Early layer gradient norms are noticeably **lower** than later layers
2. **Weak Early Updates**: Weight deltas decrease for earlier layers
3. **Gradient Ratio < 1**: First/last gradient ratio is well below 1.0
4. **Baseline Reference**: Performance is limited by poor gradient flow to early feature extractors

---
## 6. Case 2: Simple CNN with Skip Connections

Same plain CNN structure as Case 1, but with **skip connections** added.
Tests whether skip connections alone (without residual blocks) improve gradient flow.

### Expected: Early layer gradients stronger than Case 1.

### Results: CNN + Skip


**📈 Accuracy**
![](../wo_resnet_w_skip/plots/accuracy_curves.png)


**📉 Loss**
![](../wo_resnet_w_skip/plots/loss_curves.png)


**🔬 Epoch Gradients**
![](../wo_resnet_w_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../wo_resnet_w_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../wo_resnet_w_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../wo_resnet_w_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../wo_resnet_w_skip/plots/weight_deltas.png)


**🎯 Confusion Matrix**
![](../wo_resnet_w_skip/plots/confusion_matrix.png)


**🖼️ Sample Predictions**
![](../wo_resnet_w_skip/plots/sample_predictions.png)


### 🔍 Observations — CNN + Skip

1. **Improved Gradient Flow**: Skip connections visibly improve early layer gradient norms
2. **Better Ratio**: First/last gradient ratio moves closer to 1.0
3. **Stronger Updates**: Early layers now get more meaningful weight updates

> Skip connections **alone** are beneficial, even without residual block design.

---
## 7. Case 3: ResNet — WITHOUT Skip Connections

ResNet-20 architecture with `PlainBlock` (two convs per block) but **no `F(x)+x` addition**.
Isolates the question: *Is block structure alone sufficient?*

```python
def forward(self, x):  # PlainBlock
    out = self.relu(self.bn1(self.conv1(x)))
    out = self.relu(self.bn2(self.conv2(out)))  # NO + x
    return out
```

In [ ]:
# PlainBlock: ResNet shape WITHOUT skip connection
class PlainBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        # NO skip connection
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))  # ← No + x!
        return out

### Results: ResNet without Skip


**📈 Accuracy**
![](../outputs_resnet_wo_skip/plots/accuracy_curves.png)


**📉 Loss**
![](../outputs_resnet_wo_skip/plots/loss_curves.png)


**🔬 Epoch Gradients**
![](../outputs_resnet_wo_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../outputs_resnet_wo_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../outputs_resnet_wo_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../outputs_resnet_wo_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../outputs_resnet_wo_skip/plots/weight_deltas.png)


**🔢 Batch Gradient Norms**
![](../outputs_resnet_wo_skip/plots/batch_gradient_norms.png)


**🎯 Confusion Matrix**
![](../outputs_resnet_wo_skip/plots/confusion_matrix.png)


**🖼️ Predictions**
![](../outputs_resnet_wo_skip/plots/sample_predictions.png)


**🖼️ SVHN Samples**
![](../outputs_resnet_wo_skip/plots/sample_images.png)


**📋 Architecture**
![](../outputs_resnet_wo_skip/tables/table1_architecture.png)


**📋 Performance**
![](../outputs_resnet_wo_skip/tables/table2_performance.png)


**📋 Training Metrics**
![](../outputs_resnet_wo_skip/tables/table3_training_metrics.png)


**📋 Gradient Summary**
![](../outputs_resnet_wo_skip/tables/table4_gradient_summary.png)


### 🔍 Observations — ResNet (no skip)

1. **Gradient Vanishing Persists**: Removing skip addition causes same vanishing pattern
2. **Block Structure ≠ Solution**: Two-conv-per-block alone does NOT fix gradient flow
3. **Early Layer Starvation**: Gradient heatmap shows clear weakening in early layers

> **Critical**: Removing just `+ x` from ResNet blocks degrades gradient flow dramatically.

---
## 8. Case 4: ResNet WITH Skip Connections ✅

Complete ResNet-20 with **skip connections**: $y = \mathcal{F}(x) + x$

```python
def forward(self, x):  # BasicBlock
    identity = self.skip(x)                   # Skip path
    out = self.relu(self.bn1(self.conv1(x)))
    out = self.bn2(self.conv2(out))
    out += identity                            # ✅ F(x) + x
    return self.relu(out)
```

**Architecture**: Initial Conv + 3 stages × 3 BasicBlocks + AvgPool + FC = **20 layers**

In [ ]:
# BasicBlock: ResNet WITH skip connection
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        identity = self.skip(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity  # ✅ SKIP CONNECTION
        return self.relu(out)

### Results: ResNet WITH Skip


**📈 Accuracy**
![](../outputs_resnet_with_skip/plots/accuracy_curves.png)


**📉 Loss**
![](../outputs_resnet_with_skip/plots/loss_curves.png)


**🔬 Epoch Gradients**
![](../outputs_resnet_with_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../outputs_resnet_with_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../outputs_resnet_with_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../outputs_resnet_with_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../outputs_resnet_with_skip/plots/weight_deltas.png)


**🔢 Batch Gradients**
![](../outputs_resnet_with_skip/plots/batch_gradient_norms.png)


**🎯 Confusion Matrix**
![](../outputs_resnet_with_skip/plots/confusion_matrix.png)


**🖼️ Predictions**
![](../outputs_resnet_with_skip/plots/sample_predictions.png)


**🖼️ SVHN Samples**
![](../outputs_resnet_with_skip/plots/sample_images.png)


### 🔍 Observations — ResNet with Skip ✅

1. **Healthy Gradient Flow**: Gradient norms are **consistent across all layers**
2. **Uniform Updates**: All layers receive meaningful weight updates, including earliest ones
3. **Gradient Ratio ≈ 1**: First/last gradient ratio stays close to 1.0
4. **Best Accuracy**: Highest test accuracy among all four configurations
5. **Uniform Heatmap**: No stark contrast between early and late layers

> **Definitive proof**: `+ x` eliminates vanishing gradients and enables effective training.

---
# Part III: Comparative Analysis
---

## 9. Summary Comparison

| Metric | Simple CNN | CNN+Skip | ResNet(no skip) | ResNet(skip) |
|--------|:-:|:-:|:-:|:-:|
| Early layer gradient | Weak ⚠️ | Improved ✅ | Weak ⚠️ | Strong ✅ |
| Gradient ratio | ≪ 1 ❌ | ~1 | ≪ 1 ❌ | ≈ 1 ✅ |
| Early weight updates | Small | Moderate | Small | Large ✅ |
| Vanishing gradients? | Yes ❌ | Reduced | Yes ❌ | No ✅ |
| Relative accuracy | Baseline | Better | ~Baseline | Best ✅ |

### What This Proves

1. **Depth alone causes problems** (Case 1)
2. **Skip connections help any architecture** (Case 2)
3. **Block design alone is insufficient** (Case 3)
4. **Residual learning is the complete solution** (Case 4)

---
## 10. Key Takeaways & Conclusion

### Core Message

**ResNet's innovation is the skip connection**, not block structure. $y = F(x) + x$ creates a gradient highway that:
1. Prevents gradient vanishing via the identity Jacobian term
2. Makes learning identity mappings trivial ($F(x) \to 0$)
3. Enables 100+ layer networks (ResNet-152, etc.)

### Quick Reference

| Concept | Formula |
|---------|--------|
| Plain mapping | $y = \mathcal{F}(x)$ |
| Residual mapping | $y = \mathcal{F}(x) + x$ |
| Plain gradient | $\frac{\partial y}{\partial x} = \frac{\partial \mathcal{F}}{\partial x}$ |
| Residual gradient | $\frac{\partial y}{\partial x} = \frac{\partial \mathcal{F}}{\partial x} + \mathbf{I}$ |

### Viva Checklist ✅

1. Start with vanishing gradient chain-rule math
2. Define $y=F(x)$ vs $y=F(x)+x$
3. Explain why $+\mathbf{I}$ creates gradient highway
4. Show per-layer gradient norms, heatmaps, ratios
5. Conclude with first-layer vs last-layer evidence

> 💡 Strongest argument = **per-layer training dynamics**, not just top-line accuracy.

### References
- He et al. (2015). *Deep Residual Learning for Image Recognition*. CVPR 2016.
- SVHN: Netzer et al. *Reading Digits in Natural Images*. NIPS Workshop 2011.